In [ ]:
import torch
import json

beam_num = 15
dataset = "Books" # CDs_and_Vinyl, Musical_Instruments, Toys_and_Games, Video_Games, Books
test_res = torch.load(f"data/{dataset}/test_beam_{beam_num}.pt")
valid_res = torch.load(f"data/{dataset}/valid_beam_{beam_num}.pt")
train_res = torch.load(f"data/{dataset}/train_beam_{beam_num+1}.pt")

with open(f"data/{dataset}/train_extra/extra.json") as f:
    train_data = json.load(f)
with open(f"data/{dataset}/valid_extra/extra.json") as f:
    valid_data = json.load(f)
with open(f"data/{dataset}/test_extra/extra.json") as f:
    test_data = json.load(f)

print(len(train_data))
print(len(train_res["outputs"]))
print(len(valid_data))
print(len(valid_res["outputs"]))
print(len(test_data))
print(len(test_res["outputs"]))

N = len(train_data) * 1.0
n = 0
for i in range(len(train_data)):
    tgt_i = "\"" + train_data[i]["item_title"] + "\""
    intuition_list = train_res["outputs"][i]
    if tgt_i in intuition_list:
        n += 1
train_hit = n/N
print(f"train_hit: {train_hit}")

N = len(valid_data) * 1.0
n = 0
for i in range(len(valid_data)):
    tgt_i = "\"" + valid_data[i]["item_title"] + "\""
    intuition_list = valid_res["outputs"][i]
    if tgt_i in intuition_list:
        n += 1
valid_hit = n/N
print(f"valid_hit: {valid_hit}")

N = len(test_data) * 1.0
n = 0
for i in range(len(test_data)):
    tgt_i = "\"" + test_data[i]["item_title"] + "\""
    intuition_list = test_res["outputs"][i]
    if tgt_i in intuition_list:
        n += 1
test_hit = n/N
print(f"test_hit: {test_hit}")

print(f"balance_ratio: {test_hit / train_hit}")

In [ ]:
from transformers import AutoTokenizer
import torch
import json

dataset_type="test"  # train, valid, test
beam_num = 15 # If you want to use N intu items in the second stage, you should set this parameter to N for train, valid and test mode.
dataset = "Books" # CDs_and_Vinyl, Musical_Instruments, Toys_and_Games, Video_Games, Books

train_balance_ratio = {
    "CDs_and_Vinyl": {
        "5": 0.29,
        "10": 0.29,
        "15": 0.295,
        "20": 0.3,
        "25": 0.3,
        "30": 0.31
    },
    "Musical_Instruments": {
        "5": 0.4,
        "10": 0.41,
        "15": 0.43,
        "20": 0.43,
        "25": 0.44,
        "30": 0.45
    },
    "Toys_and_Games": {
        "5": 0.345,
        "10": 0.375,
        "15": 0.4,
        "20": 0.4,
        "25": 0.41,
        "30": 0.41
    },
    "Video_Games": {
        "5": 0.25,
        "10": 0.28,
        "15": 0.29,
        "20": 0.31,
        "25": 0.33,
        "30": 0.345
    },
    "Books": {
        "15": 0.15,
    }
}
lr = 3e-4

# Please fill in according to your configuration.
f1_ckpt_path = "your inturec f1 checkpoint path" 
tokenizer = AutoTokenizer.from_pretrained(f1_ckpt_path, trust_remote_code=True) 


if dataset_type == "train":
    res = torch.load(f"data/{dataset}/{dataset_type}_beam_{beam_num+1}.pt")
else:
    res = torch.load(f"data/{dataset}/{dataset_type}_beam_{beam_num}.pt")
outputs = res["outputs"]
scores = res["scores"]

with open(f"data/{dataset}/{dataset_type}_extra/extra.json") as f:
    js_data = json.load(f)


In [ ]:
import math
import random


for i in range(len(js_data)):
    row = js_data[i]
    outputs_i = outputs[i]
    scores_i = scores[i]
    tgt_i = "\"" + js_data[i]["item_title"] + "\""

    filtered_outputs = []
    filtered_scores = []
    tgt_index = -1
    for j in range(len(outputs_i)):
        out = outputs_i[j]
        score = scores_i[j]
        if out != tgt_i:
            filtered_outputs.append(out)
            filtered_scores.append(score)
        else:
            if dataset_type == 'train':
                if random.random() < train_balance_ratio[dataset][str(beam_num)]:
                        filtered_outputs.append(out)
                        filtered_scores.append(score)
                        tgt_index = j
            else:
                filtered_outputs.append(out)
                filtered_scores.append(score)
                tgt_index = j
    if dataset_type == "train":
        if len(filtered_outputs) == beam_num+1:
            index_to_remove = tgt_index
            while index_to_remove == tgt_index:
                index_to_remove = random.randint(0, len(filtered_outputs) - 1)
            filtered_outputs.pop(index_to_remove)
            filtered_scores.pop(index_to_remove)
    assert len(filtered_outputs) == beam_num
    assert len(filtered_scores) == beam_num

    js_data[i][f"intuition_list_{beam_num}"] = filtered_outputs
    js_data[i][f"intuition_scores_{beam_num}"] = filtered_scores


with open(f"data/{dataset}/{dataset_type}_extra/extra.json", "w") as f:
    json.dump(js_data, f, indent=4)
